# Voice cloning — what is actually broken

One speaker, one recording session, both languages. Four arms that separate
three things every earlier run confounded: a noisy reference, a hard
cross-lingual task, and whatever XTTS-v2 does badly on its own.

| arm | reference | speaks | answers |
|---|---|---|---|
| `en -> en` | English | English | XTTS's best case. If this fails, nothing else matters |
| `hi -> hi` | Hindi | Hindi | Monolingual Hindi — the model's ceiling |
| `en -> hi` | English | Hindi | **Production** |
| `hi -> en` | Hindi | English | Does the loss travel one way or both |

The number that makes the rest readable is none of those. It is the **real
Hindi recording scored against the anchor built from the real English one** —
same human, two languages, no synthesis. Speaker embeddings shift across
languages even for a real person, so that is the honest ceiling for `en -> hi`.
Every earlier run scored cross-lingual synthesis against a same-language
ceiling and charged the model for a gap the metric creates by itself.

Reference audio ships in the repo under `fixtures/`, so **Part 1 needs no
upload**. Part 3 is the real dubbing run and does need the bundle zip.

## 1. Check the GPU

If this prints nothing: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 2. Clone and install

`colab/requirements.txt` pins `transformers>=4.57,<5` on purpose. `coqui-tts`
imports `isin_mps_friendly`, which transformers 5.x removed, and Colab
preinstalls 5.x — so without the pin the import fails at model load. It also
pins no torch version at all, because Colab's preinstalled torch is built for
its own CUDA and replacing it breaks more than it fixes.

In [ ]:
%cd /content
!git clone -q -b test https://github.com/ayushk1233/indic-dub-pipeline.git 2>/dev/null || echo "already cloned"
%cd /content/indic-dub-pipeline
!git fetch --all --quiet && git checkout --quiet test && git pull --quiet

# IndicF5 first, this repo's pins second. IndicF5 drags numpy back to 1.x,
# which breaks transformers silently — see the note in colab/requirements.txt.
# Installing in this order lets our constraints be the ones that survive.
!pip install -q git+https://github.com/ai4bharat/IndicF5.git 2>&1 | tail -2
!pip install -q -r colab/requirements.txt 2>&1 | tail -2

# Only the packages that actually gate this run. An f5-tts numpy conflict is
# expected and documented; a numba or transformers one is not.
!pip check 2>&1 | grep -Ei "numpy|numba|transformers|tts" || echo "no relevant conflicts"
print("\ninstalled — now restart the runtime before running anything else")

## 3. Restart the runtime — not optional

The install downgraded `transformers`. Anything already imported in this
session still holds the old module objects, and the failure shows up much
later as a confusing `ImportError` inside the model loader.

**Runtime → Restart session**, then continue at section 4. Do not re-run
section 2 — the clone and the install are already on disk.

## 4. Verify the environment after the restart

In [ ]:
%cd /content/indic-dub-pipeline

import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())

# Both stacks must import in the same process, because the comparison scores
# IndicF5's output with XTTS's speaker encoder.
import TTS
print("coqui-tts", TTS.__version__)

from pathlib import Path
for name in sorted(p.name for p in Path("fixtures").glob("*")):
    size = Path("fixtures", name).stat().st_size
    print(f"  {name:<28} {size/1e6:6.2f} MB")

## 5. The experiment

Loads XTTS once, measures the calibration scale, then runs all four arms under
both greedy and sampled decoding. Roughly 6–10 minutes on a T4.

Conditioning is held at the values Coqui shipped in XTTS-v2's own
`config.json`. The earlier sweep of seven conditioning settings spanned 0.046
on a scale now known to span about 0.80, so conditioning is not a variable
worth moving here. Language is the variable under test.

In [ ]:
import importlib, colab.four_arm as four_arm
importlib.reload(four_arm)

rows = four_arm.main()

## 6. Listen

Real recordings first, so you have the target in your ears, then each arm.

Judge three things **separately** — they fail independently and have different
fixes:

1. **Identity** — is it the same person as the real recording?
2. **Humanness** — would you believe a person said it, ignoring who?
3. **Accent** — Indian or American? The similarity metric cannot see this at all.

In [ ]:
four_arm.listen(rows)

## 7. Download the report

Send me `four_arm_report.txt` along with your answers to those three questions.

In [ ]:
from google.colab import files
files.download("/content/four_arm_report.txt")

---

## 8. The real dubbing run — optional, run after Part 1

Everything above is diagnosis on isolated sentences. This synthesizes the
actual 14-segment bundle the local pipeline exported from `english.mov`.

Upload `artifacts/english_clean/tts_bundle.zip`.

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

uploaded = files.upload()

BUNDLE = Path("/content/tts_bundle")
BUNDLE.mkdir(exist_ok=True)

with zipfile.ZipFile(next(iter(uploaded)), "r") as z:
    z.extractall(BUNDLE)

print(f"\n{len(list((BUNDLE / 'request').glob('*')))} request files")
print(f"reference: {(BUNDLE / 'request' / 'reference.wav').stat().st_size/1e6:.2f} MB")

In [ ]:
from colab.xtts_worker import XTTSWorker

worker = XTTSWorker(BUNDLE)
worker.load_bundle()
worker.run()

QC report for the run — pace, speaker similarity and per-segment verdicts.

In [ ]:
import sys
for name in [n for n in list(sys.modules) if n.startswith("src.eval")]:
    del sys.modules[name]

from src.eval.harness import build_report, render_report
print(render_report(build_report(job_dir=BUNDLE, bundle_dir=BUNDLE)))

In [ ]:
import shutil
shutil.make_archive("/content/synthesis_output", "zip", BUNDLE / "output")
files.download("/content/synthesis_output.zip")

---

## What to send back

1. `four_arm_report.txt`
2. Identity / humanness / accent, in words, for each of the four arms
3. If you ran Part 8: `synthesis_output.zip` and the QC table

The decision rule is already fixed, so the numbers alone settle most of it:

- `en -> en` high → the old reference was the whole problem
- `en -> en` still low → XTTS cannot clone this voice; fine-tuning is on the table
- `hi -> hi` high but `en -> hi` low → cross-lingual gap → voice conversion, not fine-tuning
- `en -> hi` near its ceiling → identity is done; naturalness decides, and IndicF5 is next

---

## 9. XTTS-v2 against IndicF5

The four arms settled the diagnosis. This settles the choice.

XTTS-v2's weights are CPML, Coqui is gone, and no one can grant commercial
terms now — so whatever it scores, it cannot be what ships. IndicF5 is MIT,
0.4B parameters, 1417 hours of Indian speech, and clones from a reference clip
plus its transcript.

Everything is held identical: same reference clips, same seven Hindi sentences,
same anchors, same floor, and XTTS's own speaker encoder scoring both. Seven
sentences per arm rather than three, because a model difference worth acting on
could be 0.05 and three samples could not resolve that. The standard error is
printed so the gap can be read against its own noise.

In [ ]:
import importlib, colab.indicf5_check as check
importlib.reload(check)

rows = check.main()

## 10. Listen — both models, same sentence, back to back

Identity is one axis and the table already covers it. Listen for the other two:

1. **Naturalness** — does either sound synthetic?
2. **Emphasis** — does the stress land where *you* would put it? You flagged this
   on the XTTS run, and no cosine similarity can see it.

In [ ]:
check.listen(rows)

## 11. Download the report

In [ ]:
from google.colab import files
files.download("/content/indicf5_report.txt")

---

## 12. Transliteration probe — can IndicF5 speak English out of Devanagari?

Sections 9–11 settled which model clones this voice. This settles whether that
model can be handed English at all.

FINDINGS §4 recorded that IndicF5 cannot generate English from Latin text — not
accented English, not English. "Seven were impossible, and we had to rewrite
them." came back as `Sraindari ansu alwe atcho rureshi chong.` The vocabulary
was ruled out as the explanation: Latin is the largest script in its
2545-token vocabulary and the English transcript tokenizes at 100%.

But the model is fluent in Devanagari, and Hindi speech is full of English
loanwords spoken with Indian phonology — the shipping `en -> hi` run already
synthesized प्रोजेक्ट, वीडियो, इंटरव्यू and सिस्टम cleanly at 93% of scale. So spell
the English in Devanagari and see what comes out.

| arm | text handed to the model | seeds | purpose |
|---|---|---|---|
| `latin` | `Let me tell you what this project actually does.` | 1 | negative control, known to fail |
| `deva_hand` | `लेट मी टेल यू व्हाट दिस प्रोजेक्ट ऐक्चुअली डज़।` | 3 | the hypothesis, and its own noise floor |

Twenty-eight clips, about an hour, most of it model load.

`latin` **must fail.** It is regenerated rather than cited so this session
proves its own content check still fires. If it comes back clean the harness is
wrong, not the model, and nothing else in the report means anything.

Three seeds on the hypothesis arm because FINDINGS §14 records the
seven-configuration conditioning sweep as a claim that turned out wrong — its
entire span was smaller than the scale's own noise. **An arm gap is not a
result until it exceeds the seed spread**, so the spread is measured here
rather than at the end.

### 12a. Vocabulary coverage, before anything expensive

One file, no GPU, and it catches a failure that is otherwise silent: an
out-of-vocabulary character maps to index 0, index 0 is the space, and a space
is spoken as a pause. Nothing raises. In the report that is indistinguishable
from the model being unable to say the word — which is the one thing this probe
exists to measure.

`check_arms()` must come back with no missing tokens. Watch the hyphen in
`फोर्टी-सेवन` and `थर्टी-वन` in particular: no Devanagari line in any existing
fixture contains one, so nothing measured so far says whether it is in vocab.
If it is not, it degrades to the spaced form, which is harmless — but that is
worth knowing rather than assuming.

A 401 here is the gated repo, not a missing file. Re-run the `login()` cell.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("ai4bharat/IndicF5", allow_patterns=["checkpoints/vocab.txt"])

import importlib, colab.vocab_check as vc
importlib.reload(vc)

vc.main()        # the two reference transcripts
vc.check_arms()  # the text this probe actually generates

### 12b. Synthesize

The report refuses to print if no sampler call was intercepted, and separately
checks whether `fix_duration` took effect. That second check matters: IndicF5's
remote `__call__` decides which arguments it forwards, and a dropped keyword
reverts to the byte-ratio formula — which for an English reference generating
Devanagari over-allocates about 2.15x and fills the surplus with invented
speech. The durations would be wrong rather than absent.

In [ ]:
import importlib, colab.indicf5_xlit_probe as probe
importlib.reload(probe)

rows = probe.main()

### 12c. Download before listening — not after

`/content` goes with the session, and listening is the part that idles. Pulling
the audio down first is also what lets the judging outlast a free session that
expires.

In [ ]:
import shutil
from colab import workspace
from google.colab import files

shutil.make_archive("/content/xlit_probe", "zip",
                    workspace.out("indicf5_xlit_probe"))
files.download("/content/xlit_probe.zip")
files.download("/content/indicf5_xlit_probe.txt")

### 12d. Listen

His real recording of each sentence plays first, so the target is in your ears
rather than in memory.

Judge `deva_hand` on **sentences 2 to 6**. The report flags 0 and 1 as
`in_reference`: they sit inside the ten-second reference clip, so the model is
handed that audio *and* its transcript, and an arm that works only there has
not been shown to generalise.

Three questions, and they fail independently:

1. **Intelligible** — can you write down the English sentence from the audio?
2. **English at all** — or is it Hindi-sounding syllables in roughly that shape?
3. **Accent** — Indian English, or something else? No metric here can see this.

The specific thing to listen for is the end of words. English is
consonant-final and cluster-heavy; Hindi is not. If `टेक्स`, `फिट्स` and
`सेकंड्स` come out as "takes-a", "fits-a", "seconds-a", the failure is the
inherent schwa and it is fixable by writing halants — not evidence the approach
is dead.

In [ ]:
probe.listen(rows)

### 12e. The decision

| what the report shows | what happens next |
|---|---|
| `deva_hand` intelligible, no bad clips, pace ≈ 1.0, arm gap above the seed spread | build the automatic transliterators and run phase 1 |
| intelligible but not Indian-sounding | the accent metric is now worth building; nothing else until it exists |
| not intelligible | **stop.** FINETUNE_PLAN Route B. `en_to_deva.py`, `accent_scale.py` and `prosody.py` never get written |

Send back `indicf5_xlit_probe.txt`, `xlit_probe.zip`, and the three answers
above for each arm.